## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
# Internal import
import deep_learning_land_use_classification.config as config
import deep_learning_land_use_classification.evaluation as evaluation
from deep_learning_land_use_classification.dataset import get_single_label_data
import deep_learning_land_use_classification.wanddb_helpers as wh
from deep_learning_land_use_classification.early_stopping import EarlyStopper

# External imports
import torch
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb
import torch.nn as nn

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Getting data
The data is split into training (64%), validation (16%) and test (20%). Training is used for model training, validation is used to tune hyperparameters and pick the best model, and testing is used to assess the performance of the final model. Data is split using stratified sampling.

In [17]:
train_df, test_df, val_df, class_names, num_classes = get_single_label_data()

In [18]:
# Wandb initialization for experiment tracking
run = wh.init_run(
    task="single",
    architecture="dinov3-vitl16",
    num_classes=num_classes,
    loss="CrossEntropyLoss",
    epochs=50,
    batch_size=32,
    learning_rate=1e-3,
    optimizer="AdamW",
    pretrained=True,
    pretraining_dataset="sat493m",
    pretraining_source="huggingface",
    weights="facebook/dinov3-vitl16-pretrain-sat493m",
    model_name=None,
    augmentation=False,
    early_stopping=True,
    patience=2,
    min_delta=0.001
)

In [19]:
pretrained_model_name = "facebook/dinov3-vitl16-pretrain-sat493m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)

In [21]:
class SingleLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform
        self.label_to_idx = {label: i for i, label in enumerate(class_names)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform is None:
            raise ValueError("SingleLabelDataset requires a transform/processor.")
        inputs = self.transform(images=image, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        label = torch.tensor(self.label_to_idx[row["label"]], dtype=torch.long)

        return inputs, label

In [22]:
# Create datasets and dataloaders
train_dataset = SingleLabelDataset(train_df, class_names, processor)
val_dataset  = SingleLabelDataset(val_df, class_names, processor)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
val_loader  = DataLoader(val_dataset, batch_size=run.config.batch_size, shuffle=False)


In [23]:
# this is a simple wrapper around the DINO backbone to add a classification head on top
class DinoClassifier(torch.nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = torch.nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values) # get the output of the backbone
        features = outputs.pooler_output
        return self.classifier(features)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

backbone = AutoModel.from_pretrained(pretrained_model_name)

model = DinoClassifier(backbone, num_classes=num_classes).to(device)
model = model.to(device)


Using device: cuda


Loading weights: 100%|██████████| 415/415 [00:00<00:00, 3639.39it/s]


In [ ]:
criterion = nn.CrossEntropyLoss()

for p in model.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),  # only head
    lr=run.config.learning_rate,
    weight_decay=0.01
)

In [26]:
wh.log_model_summary(run, model)

### Train Model

In [27]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for inputs, labels in loader:
        pixel_values = inputs["pixel_values"].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [28]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            pixel_values = inputs["pixel_values"].to(device)
            labels = labels.to(device)

            outputs = model(pixel_values)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [30]:
epochs = run.config.epochs
early_stopper = EarlyStopper(patience=run.config.patience, min_delta=run.config.min_delta)

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss  = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"val Loss:  {val_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_singlelabel(
        model,
        val_loader,
        device,
        num_classes=num_classes,
    )
        
    wh.log_epoch(run, epoch, train_loss, val_loss,
        precision, recall, f1, p_macro, r_macro, f1_macro, class_names)
    
    # --- Early stopping check ---
    if early_stopper.step(val_loss, model):
        print(f"Early stopping triggered at epoch {epoch+1}. Best val loss: {early_stopper.best_loss:.4f}")
        break

# Restore the weights from the best epoch
early_stopper.restore_best_weights(model)
print("Restored best model weights.")

run.finish()

Epoch 1/50
Train Loss: 2.1488
val Loss:  1.4301
Epoch 2/50
Train Loss: 1.1203
val Loss:  1.0051
Epoch 3/50
Train Loss: 0.7848
val Loss:  0.8098
Epoch 4/50
Train Loss: 0.6061
val Loss:  0.7049
Epoch 5/50
Train Loss: 0.4981
val Loss:  0.6328
Epoch 6/50
Train Loss: 0.4195
val Loss:  0.5750
Epoch 7/50
Train Loss: 0.3580
val Loss:  0.5345
Epoch 8/50
Train Loss: 0.3140
val Loss:  0.5161
Epoch 9/50
Train Loss: 0.2794
val Loss:  0.4733
Epoch 10/50
Train Loss: 0.2466
val Loss:  0.4560
Epoch 11/50
Train Loss: 0.2226
val Loss:  0.4431
Epoch 12/50
Train Loss: 0.1993
val Loss:  0.4304
Epoch 13/50
Train Loss: 0.1821
val Loss:  0.4126
Epoch 14/50
Train Loss: 0.1680
val Loss:  0.3988
Epoch 15/50
Train Loss: 0.1518
val Loss:  0.3886
Epoch 16/50
Train Loss: 0.1436
val Loss:  0.3907
Epoch 17/50
Train Loss: 0.1301
val Loss:  0.3807
Epoch 18/50
Train Loss: 0.1211
val Loss:  0.3686
Epoch 19/50
Train Loss: 0.1113
val Loss:  0.3656
Epoch 20/50
Train Loss: 0.1049
val Loss:  0.3642
Epoch 21/50
Train Loss: 0.097

class/agricultural/f1,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
class/agricultural/precision,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
class/agricultural/recall,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
class/airplane/f1,▁▄▆▅▄▅▆▅▇▅▄▇█▇█▇▇█▆▅██▇█▇▇███▇▇█▆▆█▇██
class/airplane/precision,▁▆▆▅▇█████▇███████████████████████████
class/airplane/recall,▁▂▅▅▂▄▅▄▇▄▂▇█▇█▇▇█▅▄██▇█▇▇███▇▇█▅▅█▇██
class/baseballdiamond/f1,▁▃▂▆▆▆▆█▆▆█▆▆█▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆
class/baseballdiamond/precision,▃▇▁█▅▅▅█▅▅████▅███████████████████████
class/baseballdiamond/recall,▁▁▅▅███████▅▅██▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
class/beach/f1,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+59,...
